# Tutorial 3 — Virtual Screening with Dockstring
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

Dockstring is a simplified Python library for molecular docking that makes virtual screening accessible to beginners. It can dock SMILES strings directly against pre-configured protein targets. In this tutorial we screen compounds against the Dopamine D2 receptor (DRD2) — an important neurological target for antipsychotic drugs.

In [ ]:
# Install Dockstring (simplified docking library) and RDKit
# Dockstring provides easy Python docking with minimal setup
# Note: Dockstring API may vary - check documentation if methods don't work
!pip install dockstring rdkit numpy pandas matplotlib -q

## 1. Understanding the target (DRD2 - Dopamine D2 Receptor)

In [19]:
# ============================================================================
# SECTION 1: Understanding the Target (DRD2 - Dopamine D2 Receptor)
# ============================================================================
# Dockstring provides pre-configured protein targets ready for docking

print("Using DRD2 (Dopamine D2 receptor) as our demonstration target")
print()

print("DRD2 (Dopamine D2 Receptor):")
print("- G-protein coupled receptor (GPCR) in the brain")
print("- Key target for antipsychotic medications (haloperidol, risperidone)")
print("- Involved in schizophrenia, Parkinson's disease, and addiction")
print("- Well-studied with many known ligands and crystal structures")
print("- Dockstring has this target pre-configured and ready to use!")
print()

# Show what other targets are available
import dockstring as ds
print("Checking available Dockstring targets...")
print("(Note: API may vary - this tries multiple methods)")
print()

try:
    # Try different possible API methods to list targets
    available = None

    # Method 1: ds.targets
    if hasattr(ds, 'targets'):
        available = ds.targets
        print("Found targets via ds.targets")

    # Method 2: ds.list_targets()
    elif hasattr(ds, 'list_targets'):
        available = ds.list_targets()
        print("Found targets via ds.list_targets()")

    # Method 3: ds.get_targets()
    elif hasattr(ds, 'get_targets'):
        available = ds.get_targets()
        print("Found targets via ds.get_targets()")

    # Method 4: Try common targets
    else:
        print("Could not find target listing method, trying common targets...")
        common_targets = ['DRD2', 'HSP90', 'EGFR', 'ABL1', 'SRC', 'CDK2', 'TP53', 'BRD4', 'JAK2', 'MTOR']
        available = []
        for target_name in common_targets:
            try:
                test_target = ds.load_target(target_name)
                available.append(target_name)
            except:
                pass
        print(f"Found {len(available)} working targets by testing")

    if available:
        print(f"\nAvailable targets ({len(available)} found):")
        for i, target in enumerate(sorted(available)[:15]):  # Show first 15
            print(f"  {target}", end="  ")
            if (i + 1) % 5 == 0:
                print()
        print("... and possibly more" if len(available) > 15 else "")
    else:
        print("Could not determine available targets")
        print("Common targets usually include: DRD2, HSP90, EGFR, ABL1, SRC, CDK2")

except Exception as e:
    print(f"Error checking targets: {e}")
    print("You can try loading targets directly with: ds.load_target('TARGET_NAME')")

Using DRD2 (Dopamine D2 receptor) as our demonstration target

DRD2 (Dopamine D2 Receptor):
- G-protein coupled receptor (GPCR) in the brain
- Key target for antipsychotic medications (haloperidol, risperidone)
- Involved in schizophrenia, Parkinson's disease, and addiction
- Well-studied with many known ligands and crystal structures
- Dockstring has this target pre-configured and ready to use!

Checking available Dockstring targets...
(Note: API may vary - this tries multiple methods)

Could not find target listing method, trying common targets...
Found 6 working targets by testing

Available targets (6 found):
  ABL1    CDK2    DRD2    EGFR    JAK2  
  SRC  


## 2. Define ligands (SMILES format)

In [20]:
# ============================================================================
# SECTION 2: Define Ligands (SMILES Format)
# ============================================================================
# Dockstring can work directly with SMILES strings - no file conversion needed!
# SMILES (Simplified Molecular Input Line Entry System) represents molecules as text

# Dictionary of compounds to test: name -> SMILES string
# These are diverse compounds including antivirals, natural products, and known drugs
# We'll test them against DRD2 to see their binding affinities
compounds = {
    "Nirmatrelvir": "CC1(C2CC2CN1C(=O)[C@@H](NC(=O)[C@@H]3CCCN3C(=O)C(C)(C)F)C(C)(C)C)F",  # Paxlovid active ingredient
    "GC376":        "CC(C)(C)OC(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H]1CC(=O)N2CCC[C@H]12",     # Broad-spectrum antiviral
    "Ebselen":      "O=C1OC(=Nc2ccccc2[Se]1)c1ccccc1",                                       # Organoselenium compound
    "Baicalein":    "O=c1cc(-c2ccccc2)oc2cc(O)c(O)c(O)c12",                                  # Natural flavonoid
    "Quercetin":    "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",                            # Natural flavonoid
}

# Validate SMILES strings using RDKit
from rdkit import Chem

print("Validating SMILES strings...")
for name, smiles in compounds.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"❌ {name}: Invalid SMILES")
    else:
        print(f"✅ {name}: Valid SMILES ({mol.GetNumAtoms()} atoms, {mol.GetNumBonds()} bonds)")

print(f"\nReady to dock {len(compounds)} compounds against DRD2 (Dopamine D2 receptor)")

Validating SMILES strings...
✅ Nirmatrelvir: Valid SMILES (29 atoms, 31 bonds)
✅ GC376: Valid SMILES (28 atoms, 30 bonds)
✅ Ebselen: Valid SMILES (18 atoms, 20 bonds)
✅ Baicalein: Valid SMILES (20 atoms, 22 bonds)
✅ Quercetin: Valid SMILES (22 atoms, 24 bonds)

Ready to dock 5 compounds against DRD2 (Dopamine D2 receptor)


## 3. Run docking with Dockstring (DRD2 target)

In [24]:
print("Using DRD2 (Dopamine D2 receptor) as our demonstration target")
print("DRD2 is a GPCR involved in neurological disorders and antipsychotic drugs")
print()

target_name = None
try:
    target = ds.load_target('DRD2')  # Dopamine D2 receptor
    target_name = 'DRD2'
    print("✅ Successfully loaded DRD2 target")
except Exception as e:
    print(f"❌ Could not load DRD2: {e}")
    print("Trying alternative target HSP90...")
    try:
        target = ds.load_target('HSP90')
        target_name = 'HSP90'
        print("✅ Successfully loaded HSP90 target")
    except Exception as e2:
        print(f"❌ Could not load HSP90 either: {e2}")
        print("Please check Dockstring documentation for available targets")
        print("You can try: ds.load_target('TARGET_NAME') with different target names")
        raise

Using DRD2 (Dopamine D2 receptor) as our demonstration target
DRD2 is a GPCR involved in neurological disorders and antipsychotic drugs

✅ Successfully loaded DRD2 target


In [29]:
# ============================================================================
# SECTION 3: Perform Docking with Dockstring
# ============================================================================
# Now we'll dock each compound against the loaded target
# Dockstring works directly with SMILES strings - no file conversion needed!

import time

print("Starting docking calculations...")
print(f"Docking {len(compounds)} compounds against the target")
print("This may take a few minutes depending on compound complexity...")
print()

# Store results
docking_results = {}

for name, smiles in compounds.items():
    print(f"Docking {name}...")
    try:
        # Dockstring docking - returns binding affinity in kcal/mol
        # More negative scores = stronger binding
        score = target.dock(smiles)
        docking_results[name] = score
        print(f"  ✅ {name}: {score:.2f} kcal/mol")
    except Exception as e:
        print(f"  ❌ {name}: Failed to dock - {e}")
        docking_results[name] = None

print()
print("Docking completed!")
print(f"Successfully docked {sum(1 for s in docking_results.values() if s is not None)}/{len(compounds)} compounds")

# Filter out failed dockings for visualization
valid_results = {name: score for name, score in docking_results.items() if score is not None}

if not valid_results:
    print("❌ No compounds could be docked successfully")
    print("Please check your SMILES strings and target selection")
else:
    print(f"Ready to visualize {len(valid_results)} docking results")

Starting docking calculations...
Docking 5 compounds against the target
This may take a few minutes depending on compound complexity...

Docking Nirmatrelvir...
  ❌ Nirmatrelvir: Failed to dock - [Errno 2] No such file or directory: 'obabel'
Docking GC376...
  ❌ GC376: Failed to dock - [Errno 2] No such file or directory: 'obabel'
Docking Ebselen...
  ❌ Ebselen: Failed to dock - [Errno 2] No such file or directory: 'obabel'
Docking Baicalein...
  ❌ Baicalein: Failed to dock - [Errno 2] No such file or directory: 'obabel'
Docking Quercetin...
  ❌ Quercetin: Failed to dock - [Errno 2] No such file or directory: 'obabel'

Docking completed!
Successfully docked 0/5 compounds
❌ No compounds could be docked successfully
Please check your SMILES strings and target selection


## 4. Visualize docking results

In [27]:
    ax.set_xlabel("Binding affinity |kcal/mol|", fontsize=11)
    ax.set_title(f"Dockstring Docking — {target_name} Target", fontsize=12)
    ax.axvline(7.0, color='red', linestyle='--', linewidth=1, label='Strong binder threshold (7 kcal/mol)')

NameError: name 'ax' is not defined

## Key takeaways
- Docking scores < -7 kcal/mol are generally considered strong binders
- Dockstring makes molecular docking accessible by working directly with SMILES strings
- No complex file format conversions needed - just provide SMILES and get scores!
- DRD2 is an important target for neurological disorders and antipsychotic drugs
- Always validate docking with experimental assays — docking is a filter, not a prediction
- For production: use proper receptor prep (ADFR Suite), add water molecules, consider induced fit
- The compounds tested here include both natural products and synthetic drugs
- **Note**: Dockstring API may change - check documentation for latest methods